In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import numpy as np
from datetime import datetime
import random
import os
import sys
import pickle

In [2]:
tournaments_df = pd.read_csv('../Data/Production/tournaments.csv').set_index(['Name', 'Year'])
players_df = pd.read_csv('../Data/Production/players.csv').set_index('Name')

matches_df = pd.read_csv('../Data/Production/matches.csv')
matches_df = matches_df.drop(matches_df.columns[[0]], axis = 1)
df = matches_df.drop(['Tournament Name', 'Year', 'Date', 'Surface', 'Score'], axis = 1)
df

,Player 1,Player 2,Winner,Player 1 Ranking,Player 2 Ranking,Player 1 Previous Wins,Player 2 Previous Wins,P1 Surface Wins,P1 Surface Matches,P2 Surface Wins,P2 Surface Matches,P1 Last 10 Matches,P2 Last 10 Matches
0,Lleyton Hewitt,Guillermo Canas,Lleyton Hewitt,-1.0,-1.0,1.0,0.0,10.0,10.0,4.0,5.0,8.0,7.0
1,Lleyton Hewitt,Roger Federer,Lleyton Hewitt,6.0,15.0,0.0,0.0,10.0,10.0,5.0,7.0,8.0,7.0
2,Guillermo Canas,Tommy Robredo,Guillermo Canas,61.0,69.0,1.0,0.0,4.0,5.0,3.0,4.0,7.0,6.0
3,Lleyton Hewitt,Gilles Elseneer,Lleyton Hewitt,6.0,323.0,0.0,0.0,10.0,10.0,4.0,6.0,8.0,4.0
4,Roger Federer,Raemon Sluiter,Roger Federer,15.0,107.0,0.0,0.0,5.0,7.0,4.0,6.0,7.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
86551,Tatsuma Ito,Thomas Fabbiano,Tatsuma Ito,141.0,88.0,0.0,0.0,75.0,159.0,57.0,126.0,7.0,4.0
86552,Brayden Schnur,Goncalo Oliveira,Brayden Schnur,95.0,257.0,0.0,0.0,15.0,33.0,0.0,4.0,1.0,4.0
86553,Henri Laaksonen,Runhao Hua,Henri Laaksonen,110.0,685.0,0.0,0.0,24.0,60.0,0.0,2.0,4.0,0.0
86554,Yuichi Sugita,Zhao Zhao,Yuichi Sugita,129.0,1602.0,0.0,0.0,86.0,177.0,0.0,1.0,7.0,0.0


In [ ]:
p1, p2, p1_rank, p2_rank = list(df['Player 1']), list(df['Player 2']), list(df['Player 1 Ranking']), list(df['Player 2 Ranking'])
p1_prev, p2_prev = list(df['Player 1 Previous Wins']), list(df['Player 2 Previous Wins'])
p1_surface, p2_surface = list(df['P1 Surface Matches']), list(df['P2 Surface Matches'])
p1_surf_wins, p2_surf_wins = list(df['P1 Surface Wins']), list(df['P2 Surface Wins'])
p1_10, p2_10 = list(df['P1 Last 10 Matches']), list(df['P2 Last 10 Matches'])

for i in range(len(p1)):
    if random.random() > 0.5:
        temp = p1[i]
        p1[i] = p2[i]
        p2[i] = temp

        temp = p1_rank[i]
        p1_rank[i] = p2_rank[i]
        p2_rank[i] = temp
        
        temp = p1_prev[i]
        p1_prev[i] = p2_prev[i]
        p2_prev[i] = temp

        temp = p1_surface[i]
        p1_surface[i] = p2_surface[i]
        p2_surface[i] = temp

        temp = p1_surf_wins[i]
        p1_surf_wins[i] = p2_surf_wins[i]
        p2_surf_wins[i] = temp

        temp = p1_10[i]
        p1_10[i] = p2_10[i]
        p2_10[i] = temp        
        
df = pd.DataFrame({'P1' : p1, 'P2' : p2, 'Winner' : df['Winner'], 'P1 Ranking' : p1_rank, 'P2 Ranking' : p2_rank})
df['P1 Previous Wins'], df['P2 Previous Wins'], df['P1 Surface Matches'], df['P2 Surface Matches'] = p1_prev, p2_prev, p1_surface, p2_surface
df['P1 Surface Wins'], df['P2 Surface Wins'], df['P1 Last 10 Matches'], df['P2 Last 10 Matches'] = p1_surf_wins, p2_surf_wins, p1_10, p2_10
df['winner'] = df.apply(lambda x: 0 if x['Winner'] == x['P1'] else 1, axis = 1)

y = df['winner']
X = df.drop(['P1', 'P2', 'Winner', 'winner'], axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [14]:
df

,P1,P2,Winner,P1 Ranking,P2 Ranking,P1 Previous Wins,P2 Previous Wins,P1 Surface Matches,P2 Surface Matches,P1 Surface Wins,P2 Surface Wins,P1 Last 10 Matches,P2 Last 10 Matches,winner
0,Lleyton Hewitt,Guillermo Canas,Lleyton Hewitt,-1.0,-1.0,1.0,0.0,10.0,5.0,10.0,4.0,8.0,7.0,0
1,Roger Federer,Lleyton Hewitt,Lleyton Hewitt,15.0,6.0,0.0,0.0,7.0,10.0,5.0,10.0,7.0,8.0,1
2,Guillermo Canas,Tommy Robredo,Guillermo Canas,61.0,69.0,1.0,0.0,5.0,4.0,4.0,3.0,7.0,6.0,0
3,Gilles Elseneer,Lleyton Hewitt,Lleyton Hewitt,323.0,6.0,0.0,0.0,6.0,10.0,4.0,10.0,4.0,8.0,1
4,Raemon Sluiter,Roger Federer,Roger Federer,107.0,15.0,0.0,0.0,6.0,7.0,4.0,5.0,3.0,7.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86551,Thomas Fabbiano,Tatsuma Ito,Tatsuma Ito,88.0,141.0,0.0,0.0,126.0,159.0,57.0,75.0,4.0,7.0,1
86552,Goncalo Oliveira,Brayden Schnur,Brayden Schnur,257.0,95.0,0.0,0.0,4.0,33.0,0.0,15.0,4.0,1.0,1
86553,Henri Laaksonen,Runhao Hua,Henri Laaksonen,110.0,685.0,0.0,0.0,60.0,2.0,24.0,0.0,4.0,0.0,0
86554,Yuichi Sugita,Zhao Zhao,Yuichi Sugita,129.0,1602.0,0.0,0.0,177.0,1.0,86.0,0.0,7.0,0.0,0
